In [3]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
sp.init_printing()

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "Helvetica"
})

import matplotlib as mpl

In [4]:
import sys
from os.path import dirname
sys.path.append(dirname("/home/andrey/Documents/LPTMS/"\
                      + "frusa_symmetry/python/src/"))
import symmetry_utils as symmetry

In [54]:
def face_pair_interaction(*args, out_type = 'as_int', message = False):
    """
    Generates a map from pairs of face labels to interaction indices.

    Arguments:
    args     - int or array_type, provides information about particle's face 
               coloring.
               If int, gives the total number of faces, which are all assumed 
               to be distinct. Faces are given integer labels for the
               interaction map.
               If array_type, gives the labels/colors of particle faces, which
               are generally not all distinct;
    out_type - str, optional, when set to 'as_int' (default), returns the
               interaction map with integer keys, corresponding hashes of the 
               input face labels. 
               If set to 'as_label', returns the interaction map with keys of
               the same type as the input face labels;
    message  - bool, optional, default is False. When True, used to print 
               particle face labels.

    Returns:
    interaction_map - dict, map of face label pairs to interaction indices;
    n_interactions  - int, number of unique interactions, equal to 
                      (n_faces*(n_faces + 1))//2.
    """

    # Determine the type of input
    args_type = type(args[0])

    if args_type == int:
        n_faces = args[0]
        face_labels = np.arange(n_faces)

    elif args_type == list or args_type == np.ndarray:
        face_labels = np.unique(args[0])
        n_faces = len(face_labels)

    else: 
        raise TypeError('Input must be either int or array_type, not '\
                        + args_type.__name__)

    # Calculate face-contact interaction map

    n_interactions = (n_faces*(n_faces + 1))//2
    interaction_map = {}
    interaction_values = np.arange(n_interactions)    
    
    counter = 0
    
    for i1, face_1 in enumerate(face_labels):
        for i2, face_2 in enumerate(face_labels):
            if i2 > i1:
                continue

            if out_type == 'as_label':
                interaction_map[(face_1,face_2)] = counter
                interaction_map[(face_2,face_1)] = counter
                
            elif out_type == 'as_int':
                interaction_map[(i1,i2)] = counter
                interaction_map[(i2,i1)] = counter

            counter += 1

    # Sanity check for the total number of unique interactions
    if counter != n_interactions:
        warnings.warn('Incorrect number of unique interactions!'\
                    + 'Number of unique interactions is calculated to be'\
                    + '{}'.format(counter)\
                    + ' but shoud be {} instead!'.format(n_interactions))

    
    if message == True:
        print('Generating a face-contact map for a particle with '\
              + str(n_faces) + ' distict faces, labelled as')
        print(*face_labels, sep=', ')

    return interaction_map, n_interactions

In [55]:
test_fc_map, test_fc_values = face_pair_interaction(['red','blue'],out_type='as_label',message=True)

Generating a face-contact map for a particle with 2 distict faces, labelled as
blue, red


In [51]:
test_fc_map

{(np.str_('blue'), np.str_('blue')): 0,
 (np.str_('red'), np.str_('blue')): 1,
 (np.str_('blue'), np.str_('red')): 1,
 (np.str_('red'), np.str_('red')): 2}

In [52]:
test_fc_values

array([0, 1, 2])

In [14]:
x,y,z = sp.symbols('x y z')

In [23]:
#Initialize generator sets for a given point group

test_generators_symbols = symmetry.point_group_lib['D6h']['generators']
test_generators_operators = [symmetry.get_symmetry_matrix(s) \
                             for s in test_generators_symbols]
test_generators_symbols

['C3p_H,001', 'C2,001', 'C2,110', 'I,000']

In [32]:
# Calculate orbit, transporters, and generators of the stabilizer group
test_o, test_t, test_s = symmetry.point_transform(np.array([1,0,0]), test_generators_operators)

In [33]:
test_o

[array([1, 0, 0]),
 array([0, 1, 0]),
 array([-1,  0,  0]),
 array([-1, -1,  0]),
 array([ 0, -1,  0]),
 array([1, 1, 0])]

In [34]:
# Check that the transporter elements produce the full orbit by acting on the 
# first 'seed' point
for p in test_o:
    print(test_t[tuple(p)].dot(test_o[0]))

[1 0 0]
[0 1 0]
[-1  0  0]
[-1 -1  0]
[ 0 -1  0]
[1 1 0]


In [35]:
# Verify the stabilizers of the point
[symmetry.get_symmetry_symbol(s) for s in test_s]

['C2_H,100', 'm,001']

In [38]:
symmetry.point_transform(np.array([0,y,0]), test_s)

([array([0, y, 0], dtype=object), array([-y, -y, 0], dtype=object)],
 {(0,
   y,
   0): array([[1, 0, 0],
         [0, 1, 0],
         [0, 0, 1]]),
  (-y,
   -y,
   0): array([[ 1, -1,  0],
         [ 0, -1,  0],
         [ 0,  0, -1]])},
 [array([[ 1,  0,  0],
         [ 0,  1,  0],
         [ 0,  0, -1]])])

In [22]:
symmetry.permutation_representation(test_o, test_t[tuple(test_o[3])])